# 01 · Tus primeras líneas de Spark con los viajes de NYC

Este es el primero de tres notebooks de la clase 5. Aquí aprendes a manejar datos con Spark usando los datos reales del curso: qué dicen, cómo se ven, cómo se seleccionan, se transforman, se filtran, se agrupan y se unen. El segundo notebook explica qué hace Spark por dentro cuando ejecutas estas líneas, y el tercero es el lab donde escribes la ingesta de las tres fuentes a tu propio bronze.

El caso del curso es esta pregunta: ¿la lluvia cambia los viajes en NYC? Para responderla se necesitan tres fuentes de datos, y la primera sección te las presenta una por una, con un registro real de cada una. Después viene la parte práctica. Cada sección explica una operación, la ejecuta sobre los viajes de enero de 2024 y termina con un ejercicio corto, «Pruébalo tú», que resuelves antes de seguir. Antes de terminar hay una sección con las funciones de columna que usarás en el lab, y al final una sobre cómo depurar paso a paso, con tres errores puestos a propósito para que los arregles.

**Cómo se usa.** Ejecuta las celdas en orden, una por una, con Shift+Enter. Cada concepto sigue el mismo ritmo: un texto corto que dice qué es y por qué importa, una celda de código y un párrafo «Qué observar» que te dice qué mirar en la salida. Las cifras que no están en el texto las produce la celda en tu propio entorno: cuando una cifra depende de la máquina, el texto dice «lo verás al ejecutar». Cada celda de código empieza con un comentario `# Celda N`: es su número dentro de este notebook, y el texto lo usa cuando te manda a una celda concreta.

> **Antes de empezar.** En esta máquina solo cabe un proceso de Spark a la vez. Si ya abriste otro notebook de la clase y dejaste Spark encendido, ciérralo primero (cada notebook termina con una celda que apaga Spark) o reinicia el kernel del otro notebook.

## 0. Preparación: el entorno correcto, tus datos y Spark encendido

Antes de tocar datos conviene comprobar que este notebook corre con el Python del proyecto, que sabes cuál es tu bucket y que los archivos de enero están en el disco de esta máquina. Las celdas de esta sección ya vienen escritas y las ejecutas sin cambiarlas.

### 0.1 Verificar el kernel y Java

El kernel es el programa Python que ejecuta las celdas. Si eliges uno equivocado, las librerías no estarán instaladas. Elige siempre **Python (bigdata)**, que es el entorno `.venv` del proyecto. No elijas los kernels que se llaman *Glue PySpark*, *Glue Spark* o *SparkMagic*: abren una sesión de AWS Glue en la nube y esa sesión cobra mientras está abierta. La Celda 1 imprime qué Python usas y si Java está disponible, porque Spark corre sobre la máquina virtual de Java.

In [ ]:
# Celda 1
# Java: el kernel del notebook no lee ~/.bashrc, así que puede no ver el Java que instaló make setup
# en ~/.local/opt/java17. Si no está en el PATH, lo buscamos ahí y definimos JAVA_HOME antes de usar Spark.
import os, shutil
from pathlib import Path
if shutil.which("java") is None:
    for candidato in [os.environ.get("JAVA_HOME"), str(Path.home() / ".local/opt/java17")]:
        if candidato and (Path(candidato) / "bin/java").exists():
            os.environ["JAVA_HOME"] = candidato
            os.environ["PATH"] = f"{candidato}/bin:" + os.environ["PATH"]
            break

import json, re, subprocess, sys, time, contextlib, gc
from io import StringIO

print("Python:", sys.version.split()[0])
print("Ejecutable:", sys.executable)
if ".venv" not in sys.executable:
    print("AVISO: este kernel no es el del proyecto (la ruta no contiene .venv). "
          "Elige el kernel «Python (bigdata)» arriba a la derecha y reinicia el notebook.")

try:
    java = subprocess.run(["java", "-version"], capture_output=True, text=True)
    print("Java:", (java.stderr or java.stdout).splitlines()[0])
except FileNotFoundError:
    print("AVISO: no se encontró Java en el PATH; Spark no podrá arrancar.")

import pandas, pyarrow, pyspark
print("pandas", pandas.__version__, "| pyarrow", pyarrow.__version__, "| pyspark", pyspark.__version__)

### 0.2 Tu bucket, armado desde tu cuenta de AWS

Cada estudiante tiene su propio lake en S3. Su nombre no se escribe a mano: se arma con un patrón fijo, `<prefijo>-lake-<número de cuenta>-<región>`, igual que hace la función `lake_bucket` de `scripts/lib.sh` cuando corres un `make`. El número de cuenta se lo preguntamos a AWS con las credenciales del Code Editor (es lo mismo que `aws sts get-caller-identity`). La Celda 2 lo arma, lo imprime y comprueba que existe y que puedes leerlo.

In [ ]:
# Celda 2
import boto3
from botocore.exceptions import ClientError

REGION = "us-east-2"

# Raíz del repositorio: la carpeta que contiene pyproject.toml, subiendo desde donde está este notebook.
RAIZ = Path.home() / "bigdata"
for carpeta in [Path.cwd(), *Path.cwd().parents]:
    if (carpeta / "pyproject.toml").exists():
        RAIZ = carpeta
        break
sys.path.insert(0, str(RAIZ / "src"))   # así se puede hacer «from pipeline import ...»

# El prefijo viene del archivo .env del repo (lo escribe «make setup»); si no existe, es «bigdata».
prefijo = os.environ.get("PREFIX")
if not prefijo and (RAIZ / ".env").exists():
    m = re.search(r"^PREFIX=(.+)$", (RAIZ / ".env").read_text(), re.M)
    if m:
        prefijo = m.group(1).strip()
if not prefijo:
    prefijo = "bigdata"

cuenta = boto3.client("sts", region_name=REGION).get_caller_identity()["Account"]
BUCKET = f"{prefijo}-lake-{cuenta}-{REGION}"
print("Cuenta AWS :", cuenta)
print("Tu bucket  :", BUCKET)

s3 = boto3.client("s3", region_name=REGION)
try:
    s3.head_bucket(Bucket=BUCKET)
    print("Acceso al bucket: correcto")
except ClientError as e:
    print("AVISO: no se pudo leer el bucket:", e.response["Error"]["Code"],
          "- se usará el bucket público del curso para el archivo de enero.")

### 0.3 Los datos de enero, una sola vez, en el disco local

Spark lee archivos del disco de esta máquina, así que primero hay que traer el archivo de viajes de enero desde tu bucket (`bronze/_crudo/fhvhv/mes=2024-01/`, y si tu lake todavía tiene el diseño anterior, `bronze/fhvhv/mes=2024-01/`). Se guarda fuera del repositorio, en la carpeta `~/datos_clase05`, para que nunca llegue a un commit por accidente. Si la carpeta ya tiene el archivo, la celda no vuelve a descargarlo. Si tu bucket no lo tiene, usa la copia del bucket público del curso. Con las zonas pasa lo mismo, y si tampoco están en tu bucket se bajan del sitio de la TLC, que es de donde las toma `bronze.py`. El clima es un archivo diminuto que se pide directamente a Open-Meteo, igual que hará tu ingesta en el lab. La Celda 3 hace esa descarga.

In [ ]:
# Celda 3
from botocore import UNSIGNED
from botocore.config import Config
import urllib.request

DATOS = Path(os.environ.get("CLASE05_DATOS", Path.home() / "datos_clase05"))
RUTA_DIR_ENERO = DATOS / "bronze" / "fhvhv" / "mes=2024-01"
RUTA_ENERO = RUTA_DIR_ENERO / "fhvhv_tripdata_2024-01.parquet"
RUTA_ZONAS = DATOS / "bronze" / "taxi_zones" / "taxi_zone_lookup.csv"
MB = 1024 * 1024

from pipeline import bronze as bronze_pipeline   # trae URL_ZONAS, url_clima y las funciones de ingesta
from pipeline import io as pio                   # trae las rutas del lake: crudo, lotes, etc.

# El archivo crudo vive en bronze/_crudo/... (diseño nuevo). Si tu lake todavía tiene el diseño anterior
# (bronze/<fuente>/mes=.../<archivo>), la descarga prueba también esa ruta.
NOMBRE_ENERO = pio.nombre_viajes("2024-01")
CLAVES_ENERO = [pio.ruta_crudo("bronze", pio.VIAJES, NOMBRE_ENERO, "2024-01"),
                pio.ruta_anterior("bronze", pio.VIAJES, NOMBRE_ENERO, "2024-01")]
CLAVES_ZONAS = [pio.ruta_crudo("bronze", pio.ZONAS, pio.NOMBRE_ZONAS),
                pio.ruta_anterior("bronze", pio.ZONAS, pio.NOMBRE_ZONAS)]


def traer(claves, destino, url_respaldo=None):
    """Descarga a destino el primer objeto que exista entre `claves`, si destino todavía no existe.

    Busca primero en tu bucket, luego en el bucket público del curso y, si se indica, en una URL de respaldo.
    """
    if destino.exists():
        print(f"Ya existe, no se descarga: {destino}  ({destino.stat().st_size / MB:,.1f} MiB)")
        return
    destino.parent.mkdir(parents=True, exist_ok=True)
    parcial = destino.with_name(destino.name + ".parcial")   # evita dejar un archivo a medias
    t0 = time.perf_counter()
    origen = None
    for clave in claves:
        try:
            s3.download_file(BUCKET, clave, str(parcial))
            origen = f"s3://{BUCKET}/{clave}"
            break
        except ClientError:
            pass
    if origen is None and url_respaldo:
        urllib.request.urlretrieve(url_respaldo, parcial)
        origen = url_respaldo
    if origen is None:
        publico = boto3.client("s3", region_name=REGION, config=Config(signature_version=UNSIGNED))
        for clave in claves:
            try:
                publico.download_file("bigdata-udem-2026-publico", "paquete/" + clave, str(parcial))
                origen = f"s3://bigdata-udem-2026-publico/paquete/{clave}"
                break
            except ClientError:
                pass
    if origen is None:
        raise FileNotFoundError(f"No se encontró ninguno de estos objetos en tu bucket ni en el público: {claves}")
    parcial.rename(destino)
    print(f"Descargado desde {origen}\n  -> {destino}  ({destino.stat().st_size / MB:,.1f} MiB en {time.perf_counter() - t0:.1f} s)")


traer(CLAVES_ENERO, RUTA_ENERO)
traer(CLAVES_ZONAS, RUTA_ZONAS, url_respaldo=bronze_pipeline.URL_ZONAS)


# El clima de enero: un JSON pequeño que se pide a Open-Meteo con la URL que arma el pipeline.
RUTA_CLIMA = DATOS / "clima" / "open_meteo_2024-01.json"
if RUTA_CLIMA.exists():
    print(f"Ya existe, no se descarga: {RUTA_CLIMA}")
else:
    RUTA_CLIMA.parent.mkdir(parents=True, exist_ok=True)
    URL_CLIMA_ENERO = bronze_pipeline.url_clima("2024-01")
    with urllib.request.urlopen(URL_CLIMA_ENERO, timeout=60) as respuesta:
        RUTA_CLIMA.write_bytes(respuesta.read())
    print(f"Descargado desde Open-Meteo -> {RUTA_CLIMA}  ({RUTA_CLIMA.stat().st_size} bytes)")

### 0.4 Dos utilidades pequeñas

Para no repetir código, la Celda 4 define un cronómetro que imprime cuánto tardó un bloque y una función `fmt` que escribe los números grandes con espacios entre los miles, como en el resto del curso.

In [ ]:
# Celda 4
TIEMPOS = {}


@contextlib.contextmanager
def cronometro(etiqueta):
    """Mide un bloque, lo imprime y lo guarda en TIEMPOS."""
    t0 = time.perf_counter()
    yield
    TIEMPOS[etiqueta] = time.perf_counter() - t0
    print(f"[{etiqueta}] {TIEMPOS[etiqueta]:.2f} s")


def fmt(n):
    """19663930 -> '19 663 930'"""
    return f"{n:,}".replace(",", " ")

### 0.5 Arrancar Spark con la misma configuración del pipeline

No vamos a inventar una configuración aparte para el notebook: usamos la misma función `get_spark("local")` de `src/pipeline/engine.py` que usa el pipeline. Crea una sesión `local[*]`, que significa que todo vive en un solo proceso de Java: el driver y un único ejecutor, que tiene tantos hilos como núcleos tenga la máquina (dos en el Code Editor). No hay dos ejecutores: hay uno con dos hilos, y cada hilo corre una tarea a la vez. El driver tiene 2 GB de memoria, la zona horaria de la sesión queda en UTC y los reemplazos de particiones son dinámicos.

Después de arrancar, la Celda 5 imprime `defaultParallelism`, el número de tareas que Spark puede ejecutar al mismo tiempo, y el enlace a la Spark UI, la página donde Spark muestra cada trabajo, etapa y tarea mientras corre.

Dos palabras que aparecen desde aquí. La **SparkSession** es tu conexión con Spark: la celda siguiente la guarda en la variable `spark`, y todo lo que le pidas a Spark empieza por ella. Un **DataFrame** es una tabla de Spark: filas y columnas con nombre y tipo. A diferencia de una tabla de pandas, no guarda los datos en memoria; guarda el plan para leerlos y transformarlos, y Spark lo ejecuta cuando le pides un resultado (la sección 10 lo muestra).


In [ ]:
# Celda 5
# La Spark UI se ve a través del proxy del Code Editor, que agrega el prefijo /codeeditor/default/ports/4040.
# Spark necesita saberlo para armar sus enlaces, estilos e imágenes con ese prefijo; si no, la página
# carga sin diseño. Se indica antes de arrancar Spark, porque se lee al encender la JVM.
import os
os.environ["PYSPARK_SUBMIT_ARGS"] = "--conf spark.ui.proxyBase=/codeeditor/default/ports/4040 pyspark-shell"

from pyspark.sql import functions as F, DataFrame, Window
from pipeline.engine import get_spark

with cronometro("arrancar Spark"):
    spark = get_spark("local")

print("Versión de Spark :", spark.version)
print("Master           :", spark.sparkContext.master)
print("Memoria del driver:", spark.sparkContext.getConf().get("spark.driver.memory"))
print("defaultParallelism (tareas simultáneas):", spark.sparkContext.defaultParallelism)


def url_spark_ui():
    """Enlace a la Spark UI. Dentro del Code Editor pasa por el proxy del Space, igual que Airflow."""
    local = spark.sparkContext.uiWebUrl            # p. ej. http://10.0.0.5:4040
    puerto = local.rsplit(":", 1)[-1].strip("/")
    meta = Path("/opt/ml/metadata/resource-metadata.json")
    if meta.exists():
        try:
            datos = json.loads(meta.read_text())
            r = boto3.client("sagemaker", region_name=REGION).describe_space(
                DomainId=datos["DomainId"], SpaceName=datos["SpaceName"])
            base = re.match(r"https://[^/]+", r["Url"]).group(0)
            return f"{base}/codeeditor/default/ports/{puerto}/jobs/"   # /jobs/ directo: la raíz redirige mal detrás del proxy
        except Exception as e:                      # sin permiso o fuera del Code Editor
            print("(no se pudo consultar el Space:", type(e).__name__, ")")
    return f"{local}   (en el Code Editor: https://<id-del-space>.studio.{REGION}.sagemaker.aws/codeeditor/default/ports/{puerto}/)"


print("Spark UI:", url_spark_ui())

**Qué observar.** `defaultParallelism` es el número de hilos del único ejecutor de esta máquina; esa cifra manda cuántas tareas avanzan a la vez. Abre el enlace de la Spark UI en otra pestaña y déjalo abierto: la pestaña *Jobs* está vacía porque Spark todavía no ha hecho nada, y se irá llenando a medida que ejecutes celdas. La UI solo existe mientras la sesión de Spark está viva.

## 1. Conoce tus datos

Antes de escribir una sola transformación hay que saber con qué datos trabajas: qué contiene cada fuente, qué significa cada columna y dónde vive el archivo. Casi todos los errores de un pipeline empiezan por suponer algo equivocado sobre los datos, y casi todos se evitan mirando un registro real y su esquema. Esta sección presenta las tres fuentes del curso. Para cada una verás de dónde viene, un registro real completo, su esquema (los nombres y tipos de sus columnas), el significado de las columnas importantes y la ruta donde vive en el bucket.

| Fuente | Quién la publica | Formato | Tamaño | Una fila es |
|---|---|---|---|---|
| Viajes FHVHV | NYC TLC, el regulador de taxis y vehículos de alquiler de Nueva York | Parquet, un archivo por mes, 24 columnas | enero de 2024: 451 MiB y 19 663 930 filas | un viaje |
| Zonas | NYC TLC | CSV, 4 columnas | 265 filas | una zona de la ciudad |
| Clima | Open-Meteo | JSON de la API de archivo histórico | 31 días en enero | un día, con su lluvia y su temperatura |

Las tres se conectan entre sí. Cada viaje trae el número de la zona donde empezó, que apunta a una fila de la tabla de zonas, y trae la fecha en que empezó, que apunta a una fila del clima. Esas dos conexiones son los dos `JOIN` que construirás en el lab de silver de la clase 6.

### 1.1 Viajes: un archivo Parquet por mes

FHVHV significa *High Volume For-Hire Vehicles*: vehículos de alquiler con conductor de alto volumen, es decir Uber, Lyft y empresas similares. La TLC publica cada mes un archivo Parquet con todos los viajes. Parquet guarda los datos por columna y con su esquema dentro del archivo, y por eso Spark puede leer solo las columnas que pidas.

- **Origen:** `https://d37ci6vzurychx.cloudfront.net/trip-data/fhvhv_tripdata_2024-01.parquet`
- **En tu bucket:** el archivo original, intacto, vive en `s3://bigdata-lake-<CUENTA>-us-east-2/bronze/_crudo/fhvhv/mes=2024-01/fhvhv_tripdata_2024-01.parquet`, y la ingesta del pipeline deja además una copia en Parquet con cinco columnas de metadatos en `bronze/fhvhv/mes=2024-01/lote=<id_lote>/` (esa ingesta es el ejercicio 4 del lab de hoy)
- **En esta máquina:** `~/datos_clase05/bronze/fhvhv/mes=2024-01/fhvhv_tripdata_2024-01.parquet`

El primer viaje del archivo de enero, completo, es este. La tercera columna de la tabla es el diccionario de datos de la TLC explicado en lenguaje llano.

| Columna | Valor en este viaje | Qué significa |
|---|---|---|
| `hvfhs_license_num` | `HV0003` | Licencia de la empresa que prestó el servicio. HV0002 es Juno, HV0003 es Uber, HV0004 es Via y HV0005 es Lyft. |
| `dispatching_base_num` | `B03404` | Base (licencia de la TLC) que despachó el viaje. |
| `originating_base_num` | `B03404` | Base que recibió la solicitud original del viaje. |
| `request_datetime` | `2024-01-01 00:21:47` | Momento en que el pasajero pidió el viaje. |
| `on_scene_datetime` | `2024-01-01 00:25:06` | Momento en que el conductor llegó al punto de recogida. La TLC lo define para vehículos accesibles, así que en el archivo hay viajes con valor y viajes con nulo. |
| `pickup_datetime` | `2024-01-01 00:28:08` | Momento en que el pasajero subió al vehículo: empieza el viaje. |
| `dropoff_datetime` | `2024-01-01 01:05:39` | Momento en que el pasajero bajó: termina el viaje. |
| `PULocationID` | `161` | Zona de la TLC donde empezó el viaje (*pick-up*). Es una llave hacia la tabla de zonas. |
| `DOLocationID` | `158` | Zona donde terminó el viaje (*drop-off*). También apunta a la tabla de zonas. |
| `trip_miles` | `2.83` | Millas recorridas con el pasajero. |
| `trip_time` | `2251` | Duración del viaje en **segundos**. |
| `base_passenger_fare` | `45.61` | Tarifa base en dólares, antes de peajes, propina, impuestos y recargos. |
| `tolls` | `0.0` | Peajes pagados durante el viaje. |
| `bcf` | `1.25` | Aporte al Black Car Fund, el fondo de seguro de los conductores. |
| `sales_tax` | `4.05` | Impuesto de ventas del estado de Nueva York. |
| `congestion_surcharge` | `2.75` | Recargo por congestión del estado de Nueva York. |
| `airport_fee` | `0.0` | Cargo por recoger o dejar pasajeros en los aeropuertos de LaGuardia, Newark y JFK. |
| `tips` | `0.0` | Propina que el pasajero dio al conductor. |
| `driver_pay` | `40.18` | Pago total al conductor, sin peajes ni propinas y después de comisiones, recargos e impuestos. |
| `shared_request_flag` | `N` | Si el pasajero aceptó compartir el viaje (`Y` o `N`), se haya logrado o no. |
| `shared_match_flag` | `N` | Si el pasajero de verdad compartió el vehículo con otro pasajero que reservó por separado. |
| `access_a_ride_flag` | `N` | Si el viaje lo administró la MTA, la autoridad de transporte público, para personas con discapacidad. |
| `wav_request_flag` | `N` | Si el pasajero pidió un vehículo accesible para silla de ruedas. |
| `wav_match_flag` | `N` | Si el viaje se hizo en un vehículo accesible para silla de ruedas. |

Para la pregunta del curso importan sobre todo `pickup_datetime` (el día de la lluvia), `PULocationID` (la zona), `hvfhs_license_num` (la empresa), `trip_miles`, `trip_time`, `base_passenger_fare`, `tips` y `driver_pay`. Las demás columnas viajan con el dato, pero la respuesta no depende de ellas.

La Celda 6 abre el archivo con Spark y deja los datos en la variable `viajes`. Spark no lee las 19 663 930 filas para esto: solo mira el pie del archivo Parquet, donde está el esquema.

In [ ]:
# Celda 6
viajes = spark.read.parquet(str(RUTA_DIR_ENERO))   # lee solo el esquema (el pie del archivo), no las filas

print("Columnas:", len(viajes.columns))
viajes.printSchema()

**Qué observar.** El esquema lista las 24 columnas con su tipo. Los tiempos son `timestamp_ntz` (fecha y hora sin zona horaria, porque la TLC los reporta en hora local de Nueva York), las zonas son enteros, las millas y los montos son `double`, y `trip_time` es un entero largo (`long`) en segundos. Que el esquema venga dentro del archivo es una ventaja de Parquet: con un CSV tendrías que declarar o inferir cada tipo. Ahora mira un registro real, con una columna por línea:

In [ ]:
# Celda 7
viajes.limit(1).show(vertical=True, truncate=False)

**Qué observar.** La salida debe parecerse a la tabla de arriba, con los mismos nombres de columna. Fíjate en que `limit(1)` toma un solo viaje sin leer el archivo completo, y en que `vertical=True` imprime un campo por línea, que es la forma cómoda de leer registros anchos.

### 1.2 Zonas: una tabla pequeña que traduce números en nombres

Los viajes dicen «recogido en la zona 161», un número que no significa nada para quien lee el informe. La TLC publica una tabla de consulta (*lookup*) con una fila por zona de la ciudad. Es un CSV de 265 filas y 4 columnas.

- **Origen:** `https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv`
- **En tu bucket:** `s3://bigdata-lake-<CUENTA>-us-east-2/bronze/_crudo/taxi_zones/taxi_zone_lookup.csv` (el CSV intacto) y `bronze/taxi_zones/lote=<id_lote>/` (la versión en Parquet con metadatos)

Una fila real, la del aeropuerto JFK:

| Columna | Valor | Qué significa |
|---|---|---|
| `LocationID` | `132` | Identificador de la zona. Es el mismo número que traen `PULocationID` y `DOLocationID` en los viajes, y es la llave de la tabla: no se repite. |
| `Borough` | `Queens` | Distrito de la ciudad: Manhattan, Brooklyn, Queens, Bronx, Staten Island o EWR (el aeropuerto de Newark, que queda en Nueva Jersey). Algunas zonas especiales traen `Unknown` o `N/A`. |
| `Zone` | `JFK Airport` | Nombre de la zona. |
| `service_zone` | `Airports` | Zona de servicio de la TLC: `Yellow Zone`, `Boro Zone`, `Airports`, `EWR` o `N/A`. |

Un CSV no guarda tipos: todo es texto. Si no le dices a Spark que infiera o que convierta, las cuatro columnas se leen como `string`, incluso `LocationID`. Esa diferencia entre «parece un número» y «es un número» te va a dar problemas más de una vez en este notebook, y por eso la Celda 8 muestra el esquema tal como lo lee Spark.

In [ ]:
# Celda 8
zonas = spark.read.option("header", True).csv(str(RUTA_ZONAS))

zonas.printSchema()
zonas.where(F.col("LocationID") == "132").show(vertical=True, truncate=False)
print("Filas de la tabla de zonas:", zonas.count())

**Qué observar.** Las cuatro columnas son `string`, aunque `LocationID` contenga números. Por eso la comparación del filtro usa el texto `"132"` y no el número 132. La opción `header` le dice a Spark que la primera línea del CSV son los nombres de las columnas; sin ella, las columnas se llamarían `_c0`, `_c1`, y así. La tabla tiene las 265 filas que dice el texto.

### 1.3 Clima: un JSON por mes con una lista de días

Open-Meteo ofrece una API de archivo histórico: le pides un rango de fechas y unas coordenadas, y devuelve un JSON con la lluvia total y la temperatura media de cada día. La URL que usa el pipeline la arma `bronze.url_clima(mes)` con la latitud y la longitud de Nueva York y la zona horaria `America/New_York`.

- **Origen:** la URL que la Celda 3 arma con `bronze.url_clima` (`https://archive-api.open-meteo.com/v1/archive?...`).
- **En tu bucket:** el JSON intacto vive en `bronze/_crudo/clima/mes=2024-01/open_meteo_2024-01.json`, y su versión en Parquet, con una fila por día, vive en `bronze/clima/mes=2024-01/lote=<id_lote>/`. Esa segunda parte es la que construyes en el ejercicio 2 del lab de hoy.
- **En esta máquina:** `~/datos_clase05/clima/open_meteo_2024-01.json`.

La respuesta no trae una fila por día. Trae un solo objeto con tres listas paralelas dentro del bloque `daily`: la posición 0 de cada lista es el primer día, la posición 1 el segundo, y así. Esto es lo que devuelve la API, recortado a los tres primeros días de enero:

```json
{
  "latitude": 40.738136, "longitude": -74.04254,
  "timezone": "America/New_York",
  "daily_units": {"time": "iso8601", "precipitation_sum": "mm", "temperature_2m_mean": "°C"},
  "daily": {
    "time": ["2024-01-01", "2024-01-02", "2024-01-03", ...],
    "precipitation_sum": [0.0, 0.0, 0.0, ...],
    "temperature_2m_mean": [3.4, -0.0, 1.5, ...]
  }
}
```

| Campo | Qué significa |
|---|---|
| `daily.time` | La fecha de cada día, en formato ISO 8601 (`AAAA-MM-DD`). |
| `daily.precipitation_sum` | Lluvia total del día en milímetros. Incluye nieve derretida: es precipitación, no solo lluvia. |
| `daily.temperature_2m_mean` | Temperatura media del día en grados Celsius, medida a 2 metros del suelo. |

Para cruzar el clima con los viajes necesitas una fila por día. La Celda 9 lee el JSON con Spark, muestra su esquema (una estructura que contiene tres listas) y la aplana con `posexplode`, que convierte cada posición de una lista en una fila y conserva el número de la posición para buscar el mismo día en las otras dos listas.

In [ ]:
# Celda 9
clima_crudo = spark.read.option("multiLine", True).json(str(RUTA_CLIMA))   # multiLine: un solo objeto JSON en varias líneas
clima_crudo.printSchema()

# Paso 1: una fila por fecha; «i» es la posición del día dentro de las listas.
dias = clima_crudo.select(
    F.posexplode(F.col("daily.time")).alias("i", "fecha"),
    F.col("daily.precipitation_sum").alias("lluvia"),
    F.col("daily.temperature_2m_mean").alias("temp"),
)

# Paso 2: con la posición se busca el mismo día en las otras dos listas (element_at cuenta desde 1).
# Es la misma transformación que hace bronze.clima_a_filas en el pipeline.
clima = dias.select(
    F.col("fecha"),
    F.element_at("lluvia", F.col("i") + 1).cast("double").alias("lluvia_mm"),
    F.element_at("temp", F.col("i") + 1).cast("double").alias("temperatura"),
)

clima.show(5)
print("Días en el clima:", clima.count())

**Qué observar.** El esquema del JSON tiene un campo `daily` de tipo `struct` con tres campos de tipo `array`: eso es lo que significa «tres listas paralelas». Después de aplanar, la tabla `clima` tiene una fila por día (31 en enero) con tres columnas: `fecha` (todavía como texto), `lluvia_mm` y `temperatura` en grados. Esa es la forma que necesita el `JOIN` con los viajes. Si un día no tuviera dato de lluvia, la columna vendría con `null` y no se perdería la fila.

### 1.4 Las tres fuentes juntas

Ya conoces las tres. Para el resto del notebook usarás `viajes`, `zonas` y `clima`. Esta tabla resume cómo se conectan, porque cada vez que unas dos fuentes te hará falta saber qué columna con qué columna:

| Para saber | Une | Con la llave |
|---|---|---|
| El nombre y el distrito de la zona donde empezó el viaje | `viajes` con `zonas` | `PULocationID` = `LocationID` |
| La lluvia del día del viaje | `viajes` con `clima` | la fecha de `pickup_datetime` = `fecha` |

## 2. Ver los datos: `show`, `limit`, `columns` y `printSchema`

**Qué es.** Antes de transformar nada se mira. Spark ofrece pocas formas de mirar un DataFrame y cada una responde una pregunta distinta: `show` imprime filas, `limit` recorta el DataFrame sin imprimirlo, `columns` lista los nombres, `dtypes` y `printSchema` muestran los tipos y `count` cuenta las filas. **Por qué importa.** Un DataFrame no es una tabla en memoria: es un plan para producir una tabla. Mirar es la manera de comprobar que el plan produce lo que crees. Casi todo lo que vas a hacer en este notebook termina en uno de estos métodos.

Usamos primero la tabla de zonas porque es pequeña y cabe entera en pantalla.

In [ ]:
# Celda 10
zonas.show(5)                       # 5 filas, con las celdas largas cortadas
zonas.show(5, truncate=False)       # 5 filas con el texto completo de cada celda

cinco_zonas = zonas.limit(5)        # limit no imprime nada: devuelve otro DataFrame con 5 filas
print("limit devuelve un", type(cinco_zonas).__name__)

print("Columnas de zonas:", zonas.columns)
print("Tipos de zonas   :", zonas.dtypes)
print("Columnas de viajes:", len(viajes.columns), "| primeras cinco:", viajes.columns[:5])
print("Filas de viajes   :", fmt(viajes.count()))

**Qué observar.** `show(5)` corta los textos largos con `...` y `truncate=False` los muestra completos. `limit(5)` no imprime: devuelve un DataFrame nuevo, por eso hay que pedirle `show()` para verlo. `columns` es una lista de Python con los nombres y `dtypes` es una lista de pares nombre-tipo. El `count()` de los viajes es una acción que lee el archivo; fíjate en que tarda más que las demás líneas de la celda, porque es la única que necesita todos los datos. La cifra debe ser 19 663 930.

**Pruébalo tú.** Muestra las primeras 10 filas de `clima` sin cortar el texto, e imprime en una sola línea cuántas columnas tiene y cuáles son.

In [ ]:
# Celda 11
# Tu turno: escribe tu código aquí y ejecútala.

<details>
<summary>Una solución posible</summary>

```python
clima.show(10, truncate=False)
print(len(clima.columns), "columnas:", clima.columns)
```

</details>

## 3. Seleccionar columnas: `select` y `col`

**Qué es.** `select` devuelve un DataFrame nuevo con solo las columnas que nombras, en el orden en que las nombras. **Por qué importa.** Los viajes tienen 24 columnas y casi nunca necesitas todas. Seleccionar temprano hace que cada paso posterior trabaje con menos datos y, como Parquet guarda los datos por columna, Spark ni siquiera lee del disco las columnas que no pides. Para nombrar una columna hay tres formas equivalentes: el texto `"trip_miles"`, la función `F.col("trip_miles")` y el atributo `viajes["trip_miles"]`. En el curso usamos `F.col(...)` cuando la columna entra en una operación, porque funciona siempre, incluso con nombres que tienen espacios o coinciden con un método del DataFrame.

In [ ]:
# Celda 12
cuatro_columnas = viajes.select("pickup_datetime", "PULocationID", "trip_miles", "base_passenger_fare")
cuatro_columnas.show(5)

# Las tres formas de nombrar una columna producen lo mismo:
viajes.select("PULocationID").show(2)
viajes.select(F.col("PULocationID")).show(2)
viajes.select(viajes["PULocationID"]).show(2)

# drop hace lo contrario: deja todas las columnas menos las que nombras.
sin_banderas = viajes.drop("shared_request_flag", "shared_match_flag", "access_a_ride_flag",
                           "wav_request_flag", "wav_match_flag")
print("Columnas después de quitar las cinco banderas:", len(sin_banderas.columns))

**Qué observar.** Las tres formas imprimen la misma columna. `select` no modifica `viajes`: devuelve otro DataFrame, y `viajes` sigue teniendo sus 24 columnas. Lo mismo hace `drop`: el DataFrame original queda intacto y la variable `sin_banderas` guarda el resultado. Esa regla, que ningún DataFrame se modifica y cada paso devuelve uno nuevo, es la base de todo lo que sigue. Spark también ignora las mayúsculas en los nombres de columna (`pulocationid` funciona), pero un error de ortografía sí se paga, como verás en la sección de depuración.

**Pruébalo tú.** Selecciona la hora de recogida, la zona de recogida, la zona de destino y la propina de los viajes, y muestra cinco filas.

In [ ]:
# Celda 13
# Tu turno: escribe tu código aquí y ejecútala.

<details>
<summary>Una solución posible</summary>

```python
viajes.select("pickup_datetime", "PULocationID", "DOLocationID", "tips").show(5)
```

</details>

## 4. Crear y transformar columnas: `withColumn`, `cast`, `alias` y `withColumnRenamed`

**Qué es.** `withColumn("nombre", expresión)` devuelve un DataFrame con una columna más (o con la columna reemplazada, si el nombre ya existía). La expresión se construye con columnas y funciones de `pyspark.sql.functions`, que importamos como `F`: aritmética entre columnas, `F.round`, `F.to_date`, `F.when(...).otherwise(...)` para condicionales, `F.lit` para un valor fijo y `cast` para cambiar el tipo. `alias` le pone nombre a una expresión dentro de un `select`, y `withColumnRenamed` cambia el nombre de una columna que ya existe. **Por qué importa.** Los datos crudos casi nunca vienen en la forma que necesitas: los segundos hay que pasarlos a minutos, las fechas con hora hay que reducirlas al día y los números que llegan como texto hay que convertirlos. Esta es la operación con la que más vas a trabajar en silver.

La Celda 14 construye cinco columnas nuevas en cinco pasos, y cada paso tiene su propio nombre para que se pueda mirar por separado.

In [ ]:
# Celda 14
viajes_base = viajes.select("pickup_datetime", "PULocationID", "trip_miles", "trip_time", "base_passenger_fare", "tips")

# trip_time viene en segundos; la duración en minutos es una operación entre una columna y un número.
paso_1 = viajes_base.withColumn("duracion_min", F.col("trip_time") / 60)

# La fecha del día es la parte de fecha de pickup_datetime (se pierde la hora).
paso_2 = paso_1.withColumn("fecha", F.to_date("pickup_datetime"))

# De millas a kilómetros, redondeado a dos decimales.
paso_3 = paso_2.withColumn("distancia_km", F.round(F.col("trip_miles") * 1.609, 2))

# Un condicional: «sí» si hubo propina y «no» en caso contrario.
paso_4 = paso_3.withColumn("con_propina", F.when(F.col("tips") > 0, "sí").otherwise("no"))

# Cambiar el nombre de una columna que ya existe.
paso_5 = paso_4.withColumnRenamed("tips", "propina")

paso_5.show(5)
paso_5.printSchema()

**Qué observar.** Cada `withColumn` agrega una columna al final y deja las demás intactas. `duracion_min` es un `double` porque dividir por 60 produce decimales. `fecha` es de tipo `date`: sin hora. `con_propina` es texto. Y la última columna que cambió de nombre ahora se llama `propina`. Tú puedes cambiar el orden de los pasos o quitar uno y volver a ejecutar, porque cada paso se guarda en su propia variable.

Ahora el cambio de tipo. La tabla de zonas se leyó con todas las columnas como texto; la conversión a entero se hace con `cast`. La Celda 15 lo hace de dos formas: dentro de un `withColumn` que reemplaza la columna y dentro de un `select` con `alias`.

In [ ]:
# Celda 15
# Reemplazar LocationID (texto) por LocationID (entero): el nombre ya existe, así que se reemplaza.
zonas_tipadas = zonas.withColumn("LocationID", F.col("LocationID").cast("int"))
zonas_tipadas.printSchema()

# La misma conversión dentro de un select, dándole un nombre nuevo con alias.
viajes_base.select(
    F.col("trip_miles").alias("millas"),
    F.col("trip_miles").cast("int").alias("millas_enteras"),
).show(5)

**Qué observar.** En `zonas_tipadas`, `LocationID` ya es `int`, y las otras tres columnas siguen siendo texto. Usaremos `zonas_tipadas` en el resto del notebook. En el `select`, `cast("int")` **trunca** hacia cero en lugar de redondear (2.83 millas pasan a 2); si quieres redondear, primero se usa `F.round`. Un `cast` imposible, como convertir `"JFK Airport"` a entero, no falla: devuelve `null`. Es cómodo y peligroso a la vez, y es otra razón para mirar el resultado después de cada paso.

**Pruébalo tú.** Calcula la velocidad media de cada viaje en millas por hora (`trip_miles` dividido entre las horas, que son `trip_time` entre 3600) y guárdala en una columna llamada `velocidad_mph`, redondeada a un decimal. Muestra cinco filas con `trip_miles`, `trip_time` y `velocidad_mph`.

In [ ]:
# Celda 16
# Tu turno: escribe tu código aquí y ejecútala.

<details>
<summary>Una solución posible</summary>

```python
con_velocidad = viajes_base.withColumn(
    "velocidad_mph", F.round(F.col("trip_miles") / (F.col("trip_time") / 3600), 1))
con_velocidad.select("trip_miles", "trip_time", "velocidad_mph").show(5)
```

</details>

## 5. Filtrar filas: `where`, `&`, `|` y los nulos

**Qué es.** `where` (que también se escribe `filter`) devuelve solo las filas que cumplen una condición. La condición es una expresión de columnas: `F.col("PULocationID") == 132`. Para combinar condiciones se usan `&` (y), `|` (o) y `~` (no), y **cada condición va entre paréntesis**, porque en Python `&` y `|` se evalúan antes que las comparaciones. No se pueden usar las palabras `and`, `or` ni `not`, que Python reserva para valores simples y no para columnas. **Por qué importa.** Filtrar es la manera de descartar los datos que no sirven (viajes de cero millas, zonas sin dato) y de mirar un grupo concreto (los viajes desde un aeropuerto). Un filtro mal escrito no siempre falla: a veces devuelve un resultado equivocado sin avisar.

In [ ]:
# Celda 17
# Una condición simple: viajes que empiezan en la zona 132 (JFK Airport).
desde_jfk = viajes.where(F.col("PULocationID") == 132)

# Y: las dos condiciones a la vez, cada una entre paréntesis.
largos_con_propina = viajes.where((F.col("trip_miles") > 10) & (F.col("tips") > 0))

# O: los viajes que empiezan o terminan en JFK.
desde_o_hacia_jfk = viajes.where((F.col("PULocationID") == 132) | (F.col("DOLocationID") == 132))

# No: todo lo que no es Uber (HV0003).
no_uber = viajes.where(~(F.col("hvfhs_license_num") == "HV0003"))

print("Desde JFK              :", fmt(desde_jfk.count()))
print("Largos con propina     :", fmt(largos_con_propina.count()))
print("Desde o hacia JFK      :", fmt(desde_o_hacia_jfk.count()))
print("De una empresa distinta a Uber:", fmt(no_uber.count()))

**Qué observar.** Cada línea que termina en `count()` es una acción y lee el archivo, así que cada una tarda un poco. Los viajes «desde o hacia JFK» son más que los «desde JFK», porque el `|` suma los que llegan a ese aeropuerto. Los filtros se pueden guardar en una variable sin ejecutar nada: `desde_jfk` solo anota el plan, y el trabajo ocurre cuando llega el `count()`.

### Los nulos: un valor que no es igual ni distinto a nada

Un `null` significa «dato desconocido». Compararlo con cualquier cosa, incluso con otro `null`, no da verdadero ni falso sino *desconocido*, y un filtro descarta las filas desconocidas. Por eso `col != valor` **no** devuelve las filas con nulo, aunque a simple vista «un nulo es distinto de ese valor». Para encontrarlos hay que preguntar por el nulo de forma explícita con `isNull()` o `isNotNull()`.

In [ ]:
# Celda 18
total = viajes.count()
sin_llegada = viajes.where(F.col("on_scene_datetime").isNull()).count()
con_llegada = viajes.where(F.col("on_scene_datetime").isNotNull()).count()
distintos = viajes.where(F.col("on_scene_datetime") != F.col("pickup_datetime")).count()

print("Total de viajes                          :", fmt(total))
print("Sin hora de llegada del conductor (nulo) :", fmt(sin_llegada))
print("Con hora de llegada                      :", fmt(con_llegada))
print("Con llegada distinta de la recogida (!=) :", fmt(distintos))
print("Sin llegada + con llegada = total        :", sin_llegada + con_llegada == total)

**Qué observar.** Los viajes con nulo y los viajes con valor suman el total: la pregunta `isNull` / `isNotNull` reparte todas las filas. En cambio el conteo con `!=` es menor o igual que el de «con llegada», porque los viajes con nulo nunca pasan una comparación. Si en el lab quieres descartar los viajes sin zona de recogida, tienes que escribir `isNotNull()`; compararla con un valor no sirve.

**Pruébalo tú.** Cuenta los viajes que empezaron y terminaron en la misma zona (`PULocationID` igual a `DOLocationID`) y en los que además hubo propina. Imprime la cifra con `fmt`.

In [ ]:
# Celda 19
# Tu turno: escribe tu código aquí y ejecútala.

<details>
<summary>Una solución posible</summary>

```python
misma_zona = viajes.where((F.col("PULocationID") == F.col("DOLocationID")) & (F.col("tips") > 0))
print(fmt(misma_zona.count()))
```

</details>

## 6. Ordenar y quitar repetidos: `orderBy`, `distinct` y `dropDuplicates`

**Qué es.** `orderBy` ordena las filas por una o más columnas (ascendente por defecto; `F.desc("columna")` las pone de mayor a menor). `distinct` devuelve las filas sin repetir, `dropDuplicates(["a", "b"])` deja una sola fila por cada combinación de esas columnas y `F.countDistinct` cuenta los valores distintos. **Por qué importa.** Las tres operaciones obligan a Spark a comparar filas entre sí, y las filas que hay que comparar pueden estar en cualquier parte del archivo. Por eso son más costosas que `where` o `select`; en el segundo notebook verás exactamente por qué. La regla práctica es ordenar al final, sobre un resultado pequeño, y no sobre los datos crudos.

Primero un problema que ya conoces: `LocationID` es texto en el CSV, y el texto se ordena letra por letra.

In [ ]:
# Celda 20
print("Zonas ordenadas por LocationID como TEXTO:")
zonas.orderBy("LocationID").select("LocationID", "Zone").show(6)

print("Zonas ordenadas por LocationID como NÚMERO:")
zonas_tipadas.orderBy("LocationID").select("LocationID", "Zone").show(6)

print("Las tres de mayor número:")
zonas_tipadas.orderBy(F.desc("LocationID")).select("LocationID", "Borough", "Zone").show(3, truncate=False)

**Qué observar.** Como texto, el orden es 1, 10, 100, 101, 102 y así: compara carácter por carácter, y `"10"` va antes que `"2"`. Como número, el orden es el que esperas. Nadie te avisa: la celda con texto no da ningún error y devuelve un orden que parece válido. Es la razón para ejecutar `printSchema` antes de ordenar o comparar. Las dos últimas zonas, 264 y 265, son zonas especiales de la TLC: fíjate en sus nombres.

Ahora los repetidos y un orden sobre datos grandes, que se hace sobre un resultado pequeño. La Celda 21 muestra además `orderBy` con dos columnas y `dropDuplicates` con una lista de columnas, las dos formas que usa el lab.

In [ ]:
# Celda 21
# Valores distintos de una columna: ¿qué empresas hay en enero?
viajes.select("hvfhs_license_num").distinct().show()

# Cuántas zonas de recogida distintas hay.
distintas = viajes.agg(F.countDistinct("PULocationID").alias("zonas_de_recogida"))
distintas.show()

# Una fila por cada combinación origen-destino: ¿cuántos pares distintos de zonas hay?
pares = viajes.select("PULocationID", "DOLocationID").dropDuplicates()
print("Pares distintos origen-destino:", fmt(pares.count()))

# Los cinco viajes más largos: orderBy seguido de limit es mucho más barato que ordenar todo y mirar el principio.
viajes.orderBy(F.desc("trip_miles")).select("pickup_datetime", "PULocationID", "trip_miles", "trip_time").limit(5).show()

# orderBy con varias columnas: primero la zona de menor a mayor y, dentro de cada zona, la propina de mayor a menor.
# F.asc es el orden por defecto; F.desc invierte el sentido de esa columna.
viajes.select("PULocationID", "tips").orderBy(F.asc("PULocationID"), F.desc("tips")).show(5)

# dropDuplicates con una lista de columnas deja una sola fila por cada valor de esas columnas.
# Cuál de las filas repetidas se conserva no está garantizado: sirve cuando las repetidas son iguales entre sí.
una_por_empresa = viajes.select("hvfhs_license_num", "pickup_datetime").dropDuplicates(["hvfhs_license_num"])
una_por_empresa.show()

**Qué observar.** En enero aparecen dos códigos de licencia: `HV0003` (Uber) y `HV0005` (Lyft). El número de zonas de recogida es menor o igual que 265, porque hay zonas donde nadie recoge pasajeros. Los pares origen-destino son muchos más que las zonas, porque cada par es una pareja. Con dos columnas en `orderBy`, la segunda solo desempata a la primera, y `dropDuplicates(['hvfhs_license_num'])` deja una fila por empresa (dos en enero); la hora de recogida que ves es la de cualquier viaje de esa empresa. Y los cinco viajes más largos muestran algo que revisarás en el lab de silver de la clase 6: los datos crudos traen viajes de cientos de millas y de muchas horas dentro de una ciudad, y por eso silver los descarta.

**Pruébalo tú.** Muestra los cinco viajes con mayor propina (`tips`), con su hora de recogida, su zona de recogida y su propina.

In [ ]:
# Celda 22
# Tu turno: escribe tu código aquí y ejecútala.

<details>
<summary>Una solución posible</summary>

```python
viajes.orderBy(F.desc("tips")).select("pickup_datetime", "PULocationID", "tips").limit(5).show()
```

</details>

## 7. Agrupar: `groupBy` y `agg`

**Qué es.** `groupBy("columna")` agrupa las filas por los valores de una o más columnas, y `agg(...)` calcula una o varias agregaciones por cada grupo: `F.count`, `F.sum`, `F.avg`, `F.min`, `F.max`. A cada agregación se le pone nombre con `alias`. El resultado es un DataFrame con una fila por grupo. **Por qué importa.** Esto es el paso que convierte 19 millones de viajes en una tabla que cabe en pantalla, y es exactamente lo que hace gold: agrupa por día, zona y empresa. `F.count(F.lit(1))` cuenta filas: cuenta un 1 por cada fila del grupo.

In [ ]:
# Celda 23
por_empresa = (
    viajes
    .groupBy("hvfhs_license_num")
    .agg(
        F.count(F.lit(1)).alias("viajes"),
        F.round(F.sum("base_passenger_fare") / 1_000_000, 1).alias("tarifa_base_millones_usd"),
        F.round(F.avg("tips"), 2).alias("propina_promedio"),
        F.max("trip_miles").alias("viaje_mas_largo_mi"),
    )
    .orderBy(F.desc("viajes"))
)
por_empresa.show()

# Agrupar por una expresión: viajes por día de recogida.
por_dia = (
    viajes
    .groupBy(F.to_date("pickup_datetime").alias("fecha"))
    .agg(F.count(F.lit(1)).alias("viajes"))
    .orderBy("fecha")
)
por_dia.show(7)

**Qué observar.** `por_empresa` tiene una fila por empresa y cuatro columnas calculadas (la tarifa total va en millones de dólares para que se lea fácil); el orden final con `orderBy` se hace sobre esas pocas filas, que es la forma barata de ordenar. `por_dia` agrupa por una expresión (la fecha extraída de `pickup_datetime`) y no por una columna que ya existiera. Ten presente una diferencia fácil de confundir: `viajes.count()` es una **acción** que devuelve un número, mientras que `viajes.groupBy("x").count()` es una **transformación** que devuelve un DataFrame con una fila por grupo.

**Pruébalo tú.** Calcula, por zona de recogida (`PULocationID`), el número de viajes y la propina promedio, y muestra las cinco zonas con más viajes.

In [ ]:
# Celda 24
# Tu turno: escribe tu código aquí y ejecútala.

<details>
<summary>Una solución posible</summary>

```python
(viajes
 .groupBy("PULocationID")
 .agg(F.count(F.lit(1)).alias("viajes"), F.round(F.avg("tips"), 2).alias("propina_promedio"))
 .orderBy(F.desc("viajes"))
 .show(5))
```

</details>

## 8. Unir con las zonas: `join` y la cardinalidad

**Qué es.** `join` combina dos DataFrames fila a fila usando una condición, normalmente la igualdad de dos columnas. El tipo de unión decide qué pasa con las filas sin pareja: `inner` conserva solo las que la tienen en los dos lados, `left` conserva todas las del lado izquierdo (y deja `null` donde no hay pareja) y `left_anti` devuelve solo las del lado izquierdo que **no** tienen pareja. **Por qué importa.** Un `join` es la operación que más datos puede dañar sin avisar. La pregunta que hay que hacerse antes es la *cardinalidad*: cuántas filas del lado derecho corresponden a cada fila del lado izquierdo. Aquí cada viaje corresponde a una sola zona (muchos viajes, una zona), y por eso un `left join` deja los 19 663 930 viajes intactos. Si la tabla de zonas tuviera una zona repetida, cada viaje de esa zona aparecería dos veces y las sumas posteriores saldrían infladas.

In [ ]:
# Celda 25
condicion = viajes["PULocationID"] == zonas_tipadas["LocationID"]
viajes_con_zona = viajes.join(zonas_tipadas, condicion, "left")

viajes_con_zona.select("pickup_datetime", "PULocationID", "Borough", "Zone", "service_zone").show(5, truncate=False)

# Comprobación de cardinalidad: la llave de las zonas no debe repetirse y el join no debe cambiar el número de viajes.
llaves_repetidas = zonas_tipadas.groupBy("LocationID").count().where(F.col("count") > 1).count()
n_viajes = viajes.count()
n_unidos = viajes_con_zona.count()
sin_zona = viajes.join(zonas_tipadas, condicion, "left_anti").count()

print("Llaves repetidas en zonas       :", llaves_repetidas)
print("Viajes antes del join           :", fmt(n_viajes))
print("Viajes después del left join    :", fmt(n_unidos))
print("Viajes sin zona (left_anti)     :", fmt(sin_zona))
print("El join conservó todos los viajes:", n_viajes == n_unidos)

**Qué observar.** No hay llaves repetidas en `zonas_tipadas`, el `left join` conserva el mismo número de viajes y el `left_anti` dice cuántos viajes no encontraron su zona. El DataFrame resultante trae las columnas de los dos lados, incluidas `PULocationID` y `LocationID`, que valen lo mismo.

Para ver qué pasa cuando la cardinalidad se rompe, repetimos el experimento con una tabla de zonas donde el JFK aparece dos veces. Se usa una muestra de los viajes para que la celda sea rápida.

In [ ]:
# Celda 26
muestra_chica = viajes.sample(fraction=0.001, seed=7)

# Una tabla de zonas con una fila de más: JFK (132) repetido.
zonas_con_jfk_doble = zonas_tipadas.unionByName(zonas_tipadas.where(F.col("LocationID") == 132))
condicion_muestra = muestra_chica["PULocationID"] == zonas_tipadas["LocationID"]
condicion_doble = muestra_chica["PULocationID"] == zonas_con_jfk_doble["LocationID"]

normal = muestra_chica.join(zonas_tipadas, condicion_muestra, "left").count()
con_error = muestra_chica.join(zonas_con_jfk_doble, condicion_doble, "left").count()
viajes_de_jfk = muestra_chica.where(F.col("PULocationID") == 132).count()

print("Filas con la tabla correcta :", fmt(normal))
print("Filas con JFK repetido      :", fmt(con_error))
print("Viajes de JFK en la muestra :", fmt(viajes_de_jfk))
print("Las filas de más son exactamente los viajes de JFK:", con_error - normal == viajes_de_jfk)

**Qué observar.** Con una sola fila repetida en la tabla pequeña, el resultado crece en exactamente los viajes de esa zona. No hubo ningún error: el `join` hizo lo que le pediste. Por eso, antes de unir, se comprueba que la llave del lado pequeño sea única, como en la celda anterior. Esta comprobación vuelve en el lab de silver de la clase 6.

Por último, el `join` y el `groupBy` juntos responden una pregunta de negocio: ¿cuántos viajes empiezan en cada distrito?

In [ ]:
# Celda 27
por_distrito = (
    viajes_con_zona
    .groupBy("Borough")
    .agg(F.count(F.lit(1)).alias("viajes"))
    .orderBy(F.desc("viajes"))
)
por_distrito.show()

**Qué observar.** Una fila por distrito, ordenada de mayor a menor. La fila `N/A` corresponde a viajes cuya zona de recogida es la 265, `Outside of NYC`, una de las zonas especiales del final de la tabla, y `EWR` es el aeropuerto de Newark, que queda en Nueva Jersey. Existen en los datos y por eso aparecen, aunque sean muy pocos.

**Pruébalo tú.** Cuenta los viajes por **zona de destino** (`DOLocationID`) y muestra las cinco zonas con más llegadas, con su nombre (`Zone`). Pista: la condición del `join` usa `DOLocationID`.

In [ ]:
# Celda 28
# Tu turno: escribe tu código aquí y ejecútala.

<details>
<summary>Una solución posible</summary>

```python
(viajes
 .join(zonas_tipadas, viajes["DOLocationID"] == zonas_tipadas["LocationID"], "left")
 .groupBy("Zone")
 .agg(F.count(F.lit(1)).alias("llegadas"))
 .orderBy(F.desc("llegadas"))
 .show(5, truncate=False))
```

</details>

## 9. Lo mismo en Spark SQL

**Qué es.** Spark entiende SQL. Para consultar un DataFrame con SQL se registra con un nombre, una *vista temporal* (`createOrReplaceTempView`), y se escribe la consulta con `spark.sql(...)`, que devuelve otro DataFrame. **Por qué importa.** Quien ya sabe SQL puede empezar a trabajar con Spark desde el primer día, y los dos estilos se pueden mezclar en el mismo programa. Las dos formas producen el mismo plan de ejecución, así que ninguna es más rápida: elige la que se lea mejor. En el pipeline usamos la API de DataFrames porque cada paso es una función de Python que se puede probar por separado.

| SQL | API de DataFrames |
|---|---|
| `SELECT col1, col2` | `.select("col1", "col2")` |
| `WHERE cond1 AND cond2` | `.where(cond1 & cond2)` |
| `GROUP BY a` con `COUNT(*)` | `.groupBy("a").agg(F.count(F.lit(1)))` |
| `ORDER BY x DESC` | `.orderBy(F.desc("x"))` |
| `LEFT JOIN t ON ...` | `.join(t, condición, "left")` |
| `SELECT DISTINCT` | `.distinct()` |
| `LIMIT n` | `.limit(n)` |
| `CASE WHEN c THEN a ELSE b END` | `F.when(c, a).otherwise(b)` |

La Celda 29 repite la consulta de viajes por distrito con SQL y comprueba que da lo mismo que la versión de DataFrames de la sección anterior.

In [ ]:
# Celda 29
viajes.createOrReplaceTempView("viajes")
zonas_tipadas.createOrReplaceTempView("zonas")

por_distrito_sql = spark.sql("""
    SELECT z.Borough AS Borough, COUNT(*) AS viajes
    FROM viajes v
    LEFT JOIN zonas z ON v.PULocationID = z.LocationID
    GROUP BY z.Borough
    ORDER BY viajes DESC
""")
por_distrito_sql.show()

sobran_en_sql = por_distrito_sql.exceptAll(por_distrito).count()
sobran_en_df = por_distrito.exceptAll(por_distrito_sql).count()
print("Filas que están en uno y no en el otro:", sobran_en_sql + sobran_en_df)

**Qué observar.** La tabla de SQL es igual a la de la sección 8 y la última línea imprime cero filas de diferencia (`exceptAll` devuelve las filas que están en un DataFrame y no en el otro, y se calcula en las dos direcciones). Fíjate en que `viajes` y `zonas` son nombres de la vista; no son variables de Python.

**Pruébalo tú.** Escribe en SQL la consulta que devuelve las cinco zonas de recogida (con su nombre `Zone`) con más viajes, y muéstrala.

In [ ]:
# Celda 30
# Tu turno: escribe tu código aquí y ejecútala.

<details>
<summary>Una solución posible</summary>

```python
spark.sql("""
    SELECT z.Zone, COUNT(*) AS viajes
    FROM viajes v
    LEFT JOIN zonas z ON v.PULocationID = z.LocationID
    GROUP BY z.Zone
    ORDER BY viajes DESC
    LIMIT 5
""").show(truncate=False)
```

</details>

## 10. Transformaciones y acciones

**Qué es.** Todo lo que has escrito hasta aquí se divide en dos grupos. Las *transformaciones* describen un cálculo y devuelven un DataFrame nuevo, sin leer ni un dato. Las *acciones* piden un resultado y obligan a Spark a ejecutar todo lo que se anotó. **Por qué importa.** Esa separación es lo que permite a Spark ver el plan completo antes de empezar a ejecutar, y es la causa de dos comportamientos que sorprenden: una transformación pesada «tarda cero segundos» cuando la defines, y los errores de ejecución solo aparecen en la línea de la acción. En el segundo notebook verás por dentro cómo se aprovecha esto; aquí basta con saber distinguirlas.

| Transformaciones (devuelven un DataFrame, no ejecutan nada) | Acciones (ejecutan y devuelven un resultado) |
|---|---|
| `select`, `drop`, `withColumn`, `withColumnRenamed` | `show` |
| `where` / `filter` | `count` |
| `groupBy(...).agg(...)` y `groupBy(...).count()` | `collect`, `take`, `first`, `head` |
| `orderBy`, `distinct`, `dropDuplicates`, `limit`, `sample` | `toPandas` |
| `join`, `union`, `unionByName` | `write` (escribir un archivo o una tabla) |

In [ ]:
# Celda 31
with cronometro("definir 1: leer el archivo"):
    v = spark.read.parquet(str(RUTA_DIR_ENERO))
with cronometro("definir 2: where"):
    v = v.where(F.col("PULocationID") == 132)
with cronometro("definir 3: select"):
    v = v.select("pickup_datetime", "base_passenger_fare")
with cronometro("definir 4: withColumn"):
    v = v.withColumn("tarifa_centavos", F.col("base_passenger_fare") * 100)

print("\nHasta aquí no se ha leído ninguna fila de datos; solo se anotó el plan.\n")

with cronometro("acción: count()"):
    n = v.count()
print("Viajes que salen de la zona 132:", fmt(n))

**Qué observar.** Las cuatro líneas de «definir» tardan fracciones de segundo y el `count()` tarda lo que cuesta leer el archivo. En nuestra medición en el Code Editor, definir los cuatro pasos tardó entre 0.01 y 0.22 segundos y el `count()` tardó 1.58 segundos; en tu máquina lo verás al ejecutar. Spark tampoco devolvió ningún dato por las transformaciones: `v` es un DataFrame, un plan, no una tabla.

Una aclaración sobre los errores. Spark revisa los nombres de columnas y los tipos en cuanto defines el paso, sin leer datos, así que una columna mal escrita falla en la línea donde la escribes. Lo que solo falla en la acción son los errores que dependen de los datos o de los archivos.

**Pruébalo tú.** Antes de ejecutar la Celda 32, escribe en un comentario cuáles de sus cuatro líneas crees que van a tardar y cuáles no. Después ejecútala y compara tu predicción con los tiempos que imprime.

In [ ]:
# Celda 32
# Mi predicción: (escribe aquí cuáles líneas crees que tardan)

with cronometro("a"):
    a = viajes.where(F.col("tips") > 0)
with cronometro("b"):
    b = a.groupBy("PULocationID").agg(F.count(F.lit(1)).alias("viajes"))
with cronometro("c"):
    c = b.orderBy(F.desc("viajes"))
with cronometro("d"):
    d = c.limit(3).collect()

<details>
<summary>Una solución posible</summary>

```python
# Las líneas a, b y c son transformaciones y tardan fracciones de segundo.
# La línea d tiene collect(), una acción, y es la única que ejecuta el plan completo.
```

</details>

## 11. `show`, `take`, `collect` y `toPandas`: qué traen al driver y cuánto cuesta

**Qué es.** Las acciones que devuelven datos a Python los reúnen primero en el *driver*, el proceso de Java que coordina a Spark y que en esta máquina tiene 2 GB, y después los entregan al proceso de Python de tu notebook. **Por qué importa.** Spark puede procesar archivos mucho más grandes que su memoria porque los reparte en pedazos, pero en cuanto pides los datos de vuelta a Python todo se junta en un solo proceso. Traer un resultado pequeño no cuesta nada; traer filas crudas cuesta memoria y tiempo proporcionales al número de filas, y puede tumbar el kernel.

| Método | Qué devuelve | Dónde queda el dato | Cuándo usarlo |
|---|---|---|---|
| `show(n)` | Nada: imprime n filas (20 si no pones n). | Trae a Python solo esas filas, ya convertidas en texto. | Mirar un DataFrame. |
| `take(n)`, `head(n)`, `first()` | Una lista de objetos `Row` con n filas. | Memoria del driver. | Inspeccionar unas pocas filas desde Python. |
| `collect()` | Una lista de `Row` con **todas** las filas. | Memoria del driver. | Solo resultados pequeños. |
| `toPandas()` | Un DataFrame de pandas con **todas** las filas. | Los datos pasan por el driver y terminan en el proceso de Python, fuera de los 2 GB de Java. | Resultados pequeños que vas a analizar o graficar con pandas. |
| `limit(n)` | Otro DataFrame de Spark con n filas. | No trae nada. | Recortar antes de seguir transformando. |

Regla de ingeniería: **al driver se traen resultados, no datos.** Agrega, filtra o toma una muestra en Spark y solo entonces trae el resultado.

In [ ]:
# Celda 33
# take devuelve objetos Row: se leen por nombre, como un diccionario.
tres = zonas_tipadas.take(3)
primera = tres[0]
print("Tipo:", type(primera).__name__, "| zona:", primera["Zone"], "| como diccionario:", primera.asDict())

# collect de un resultado pequeño: el resumen por distrito.
with cronometro("collect del resumen por distrito"):
    filas = por_distrito.collect()
print("Filas traídas:", len(filas))

# toPandas de un resumen pequeño: las zonas con su número de viajes.
resumen_zonas = viajes.groupBy("PULocationID").agg(F.count(F.lit(1)).alias("viajes"))
with cronometro("toPandas del resumen por zona"):
    resumen_pd = resumen_zonas.toPandas()
print("Filas en pandas:", len(resumen_pd))

# toPandas de datos crudos: un millón de filas de dos columnas.
with cronometro("toPandas de un millón de filas crudas"):
    un_millon = viajes.select("PULocationID", "base_passenger_fare").limit(1_000_000).toPandas()
print(f"Memoria del millón de filas en pandas: {un_millon.memory_usage(deep=True).sum() / MB:,.1f} MB")

**Qué observar.** Compara los dos `toPandas`. En nuestra medición en el Code Editor, traer un millón de filas de dos columnas tardó 8.6 segundos y ocupó 11.4 MB; el resumen de 262 zonas, calculado desde cero y traído, tardó 2.0 segundos, y traerlo cuando ya estaba calculado, 0.08 segundos (16 nov. 2026); en tu máquina lo verás al ejecutar. El millón de filas son dos columnas y apenas el 5 % del mes: con las 24 columnas y las 19 663 930 filas el tiempo y la memoria se multiplican hasta que el kernel se cae. La diferencia entre las dos líneas es el argumento de toda la sección: el resumen ya trae el trabajo hecho y por eso es pequeño.

**Pruébalo tú.** Trae a pandas el resumen de viajes por distrito (`por_distrito`) y agrega con pandas una columna `porcentaje` con la participación de cada distrito en el total, redondeada a un decimal.

In [ ]:
# Celda 34
# Tu turno: escribe tu código aquí y ejecútala.

<details>
<summary>Una solución posible</summary>

```python
distritos_pd = por_distrito.toPandas()
distritos_pd["porcentaje"] = (100 * distritos_pd["viajes"] / distritos_pd["viajes"].sum()).round(1)
print(distritos_pd.to_string(index=False))
```

</details>

## 12. Funciones de columna que usarás en el lab

**Qué es.** En el lab de ingesta de esta clase y en el de silver y gold de la clase 6 completas ejercicios con unas pocas funciones de columna que todavía no has usado. Cada una recibe columnas y devuelve una columna nueva; la única excepción es `F.broadcast`, que recibe un DataFrame y devuelve otro. **Por qué importa.** Si las conoces antes, en los labs solo te queda pensar la lógica del pipeline y no buscar sintaxis. Esta sección te las presenta una por una, con un ejemplo corto sobre los viajes y un «Pruébalo tú»; el enunciado de cada ejercicio dice cuál usar y en qué hueco.

| Función | Qué hace | Dónde la usas |
|---|---|---|
| `F.coalesce(a, b)` | Devuelve, fila por fila, la primera columna que no es nula. | Lab de ingesta (clase 5), ejercicio 1; lab de silver y gold (clase 6), ejercicio 1 |
| `F.lit(valor)` | Convierte un valor fijo de Python en una columna que lo repite en todas las filas. | Lab de ingesta (clase 5), ejercicio 1; lab de silver y gold (clase 6), ejercicios 1, 3 y 4 |
| `F.concat_ws(sep, a, b)` | Une textos poniendo un separador entre ellos, y se salta los nulos. | Lab de ingesta (clase 5), ejercicio 1 |
| `F.sha2(c, 256)` | Calcula la huella SHA-256 de un texto. | Lab de ingesta (clase 5), ejercicio 1 |
| `F.concat(a, b)` | Une textos sin separador; si alguno es nulo, el resultado es nulo. | Lab de silver y gold (clase 6), ejercicio 3 |
| `F.broadcast(df)` | Pide copiar un DataFrame pequeño a cada ejecutor antes de un `join`. | Lab de silver y gold (clase 6), ejercicio 3 |
| `dropDuplicates([...])` y `orderBy` | Dejan una fila por cada valor de una llave y ordenan las filas. | Lab de silver y gold (clase 6), ejercicio 2 (`dropDuplicates`) y la comparación final (`orderBy`) |

### 12.1 `F.coalesce` y `F.lit`: reemplazar nulos y fijar valores

`F.coalesce` mira sus columnas de izquierda a derecha y devuelve la primera que no es nula: es la forma de decirle a Spark «si falta, usa esto otro». `F.lit` crea una columna con un valor fijo, y lo necesitas porque `coalesce` solo recibe columnas, no valores sueltos. Si le pasas una fecha con hora de Python, `F.lit` produce una columna de tipo `timestamp`, que es como el lab marca el momento de la ingesta. El ejemplo usa `on_scene_datetime`, que viene nula en muchos viajes, como contó la Celda 18.

In [ ]:
# Celda 35
from datetime import datetime, timezone

viajes_sin_llegada = viajes.where(F.col("on_scene_datetime").isNull())

# coalesce toma la primera columna que no es nula: la hora de llegada si existe y, si no, la de recogida.
# El segundo coalesce reemplaza el nulo por un texto fijo, creado con F.lit.
hora_de_referencia = F.coalesce(F.col("on_scene_datetime"), F.col("pickup_datetime"))
llegada_como_texto = F.coalesce(F.col("on_scene_datetime").cast("string"), F.lit("<nulo>"))

viajes_sin_llegada.select(
    "pickup_datetime",
    "on_scene_datetime",
    hora_de_referencia.alias("hora_de_referencia"),
    llegada_como_texto.alias("llegada_como_texto"),
).show(3, truncate=False)

# F.lit también recibe una fecha con hora de Python y la deja como timestamp.
momento_de_ingesta = datetime.now(timezone.utc).replace(microsecond=0)
viajes.select(F.lit(momento_de_ingesta).alias("_fecha_ingesta")).printSchema()

**Qué observar.** En las tres filas `on_scene_datetime` es nula. `hora_de_referencia` toma entonces la hora de recogida y `llegada_como_texto` muestra `<nulo>`; sin `coalesce`, esas columnas seguirían nulas. La última línea imprime el esquema con `_fecha_ingesta` de tipo `timestamp`: un valor de Python se convirtió en una columna.

**Pruébalo tú.** Crea una columna `llegada_o_recogida` con `F.coalesce` entre `on_scene_datetime` y `pickup_datetime`, y cuenta cuántas filas quedan con nulo en ella. El resultado debe ser cero.

In [ ]:
# Celda 36
# Tu turno: escribe tu código aquí y ejecútala.

<details>
<summary>Una solución posible</summary>

```python
con_hora = viajes.withColumn(
    "llegada_o_recogida", F.coalesce(F.col("on_scene_datetime"), F.col("pickup_datetime")))
print(fmt(con_hora.where(F.col("llegada_o_recogida").isNull()).count()))
```

</details>

### 12.2 `F.concat_ws`: unir textos con un separador

`F.concat_ws(separador, a, b, ...)` une varias columnas de texto en una sola y pone el separador entre ellas. Lo que debes recordar es qué hace con los nulos: **los salta**, sin dejar hueco y sin separador. Eso es cómodo para armar un nombre y peligroso para una huella, porque dos filas distintas pueden terminar con el mismo texto unido. Por eso el ejercicio 1 del lab de ingesta reemplaza los nulos con `coalesce` antes de unir. Si tienes las columnas en una lista de Python, la desempaquetas con `*`: `F.concat_ws("||", *lista)`. La Celda 37 muestra las dos situaciones.

In [ ]:
# Celda 37
# Une la zona de recogida y la de destino con una flecha. Las dos son enteros, así que primero pasan a texto.
trayecto = F.concat_ws(" -> ", F.col("PULocationID").cast("string"), F.col("DOLocationID").cast("string"))
viajes.select("PULocationID", "DOLocationID", trayecto.alias("trayecto")).show(3)

# concat_ws salta los nulos: donde on_scene_datetime es nula, el resultado es solo la hora de recogida,
# sin separador ni hueco. El nulo desaparece sin dejar rastro.
columnas_como_texto = [F.col("on_scene_datetime").cast("string"), F.col("pickup_datetime").cast("string")]
unidas = F.concat_ws("||", *columnas_como_texto)    # el * desempaqueta la lista en argumentos sueltos
viajes_sin_llegada.select("on_scene_datetime", "pickup_datetime", unidas.alias("unidas")).show(3, truncate=False)

**Qué observar.** En la primera tabla la flecha aparece entre los dos números. En la segunda, `unidas` es solo la hora de recogida: no hay `||` ni hueco que indique que faltaba la hora de llegada. Ese es el motivo de reemplazar los nulos por un marcador antes de unir.

**Pruébalo tú.** Une `hvfhs_license_num`, `PULocationID` y `DOLocationID` con el separador `|` en una columna llamada `llave_viaje`, y muestra tres filas.

In [ ]:
# Celda 38
# Tu turno: escribe tu código aquí y ejecútala.

<details>
<summary>Una solución posible</summary>

```python
llave = F.concat_ws("|", F.col("hvfhs_license_num"),
                    F.col("PULocationID").cast("string"), F.col("DOLocationID").cast("string"))
viajes.select(llave.alias("llave_viaje")).show(3, truncate=False)
```

</details>

### 12.3 `F.sha2`: la huella de un texto

`F.sha2(columna, 256)` calcula la huella SHA-256 de un texto: una cadena de 64 caracteres hexadecimales. Tiene dos propiedades que la hacen útil para encontrar filas repetidas. La primera, el mismo texto siempre da la misma huella. La segunda, un cambio mínimo en el texto da una huella completamente distinta. Recibe texto, no números: si la columna es numérica, la conviertes antes con `cast("string")`. El segundo argumento es el tamaño en bits; en el curso siempre es 256. La Celda 39 primero compara tres textos y luego calcula la huella de una fila de viajes.

In [ ]:
# Celda 39
# Tres textos: los dos primeros son idénticos y el tercero cambia solo una mayúscula.
textos = spark.createDataFrame([("JFK Airport",), ("JFK Airport",), ("JFK airport",)], "zona string")
huellas = textos.select("zona", F.sha2(F.col("zona"), 256).alias("huella"))
huellas.show(truncate=False)

# La huella de una fila de viajes: primero se une la fila en un solo texto y después se le aplica sha2.
fila_unida = F.concat_ws(
    "||", F.col("PULocationID").cast("string"), F.col("DOLocationID").cast("string"), F.col("trip_miles").cast("string"))
huellas_de_viajes = viajes.select(F.sha2(fila_unida, 256).alias("huella_del_viaje"))
huellas_de_viajes.withColumn("largo", F.length("huella_del_viaje")).show(2, truncate=False)

**Qué observar.** Los dos primeros textos tienen la misma huella, y el tercero, que solo cambia una letra, tiene otra completamente distinta. Todas las huellas miden 64 caracteres sin importar qué tan largo sea el texto de origen. Por eso comparar dos filas se reduce a comparar un solo texto de 64 caracteres.

**Pruébalo tú.** Calcula la huella SHA-256 del nombre de cada zona (`Zone`) de `zonas_tipadas` y cuenta cuántas huellas distintas hay. Compara esa cifra con el número de nombres distintos.

In [ ]:
# Celda 40
# Tu turno: escribe tu código aquí y ejecútala.

<details>
<summary>Una solución posible</summary>

```python
con_huella = zonas_tipadas.withColumn("huella", F.sha2(F.col("Zone"), 256))
huellas_distintas = con_huella.select("huella").distinct().count()
nombres_distintos = zonas_tipadas.select("Zone").distinct().count()
print(huellas_distintas, "huellas distintas y", nombres_distintos, "nombres distintos")
```

</details>

### 12.4 `F.concat`: unir textos y qué pasa con los nulos

`F.concat(a, b, ...)` une columnas de texto sin separador. A diferencia de `concat_ws`, **si alguna de las columnas es nula, el resultado es nulo**. En el ejercicio 3 del lab de silver y gold de la clase 6 lo usas para armar un nombre genérico como «Zona 9999», con un texto fijo hecho con `F.lit` y el identificador convertido a texto. Después lo combinas con `F.coalesce` para usar ese nombre genérico solo cuando falta el nombre de verdad. La Celda 41 lo prueba con una tabla de dos zonas, una con nombre y otra sin él.

In [ ]:
# Celda 41
# Una tabla pequeña con una zona que sí tiene nombre y otra que no.
zonas_de_ejemplo = spark.createDataFrame([(132, "JFK Airport"), (9999, None)], "id int, nombre string")

nombre_generico = F.concat(F.lit("Zona "), F.col("id").cast("string"))      # «Zona 132», «Zona 9999»
concat_con_nulo = F.concat(F.lit("Zona "), F.col("nombre"))                   # nulo si el nombre es nulo
nombre_o_generico = F.coalesce(F.col("nombre"), nombre_generico)              # el nombre; si falta, el genérico

zonas_de_ejemplo.select(
    "id", "nombre",
    nombre_generico.alias("generico"),
    concat_con_nulo.alias("concat_con_nulo"),
    nombre_o_generico.alias("nombre_o_generico"),
).show(truncate=False)

**Qué observar.** La zona 132 conserva su nombre y la 9999 queda como «Zona 9999», porque `coalesce` recurre al nombre genérico solo cuando el de verdad es nulo. La columna `concat_con_nulo` muestra por qué hace falta ese cuidado: unir con un nulo devuelve nulo.

**Pruébalo tú.** Sobre `zonas_tipadas`, crea una columna `etiqueta` con el formato `Borough - Zone` (por ejemplo, `Queens - JFK Airport`) usando `F.concat` y `F.lit`, y muestra cinco filas.

In [ ]:
# Celda 42
# Tu turno: escribe tu código aquí y ejecútala.

<details>
<summary>Una solución posible</summary>

```python
etiqueta = F.concat(F.col("Borough"), F.lit(" - "), F.col("Zone"))
zonas_tipadas.select("LocationID", etiqueta.alias("etiqueta")).show(5, truncate=False)
```

</details>

### 12.5 `F.broadcast`: pedir la copia de una tabla pequeña en un `join`

`F.broadcast(df)` no calcula nada. Le pide a Spark que copie el DataFrame pequeño a cada ejecutor y que resuelva el `join` dentro de cada partición del lado grande, sin mover los viajes por un shuffle. Se escribe alrededor de la tabla pequeña, dentro del `join`. Con una tabla de 265 zonas, Spark lo haría solo porque estima que pesa menos de 10 MB; en el ejercicio 3 del lab de silver y gold, la prueba apaga esa decisión automática para comprobar que **tú** lo pediste. La Celda 43 imprime las líneas del plan que muestran el broadcast; el segundo notebook explica por qué ahorra un shuffle.

In [ ]:
# Celda 43
muestra_de_viajes = viajes.sample(fraction=0.001, seed=3)
condicion_destino = muestra_de_viajes["DOLocationID"] == zonas_tipadas["LocationID"]

# F.broadcast envuelve la tabla pequeña (zonas), nunca la grande (viajes).
con_broadcast = muestra_de_viajes.join(F.broadcast(zonas_tipadas), condicion_destino, "left")

# explain() imprime el plan; aquí se captura el texto para mostrar solo las líneas del join.
salida_plan = StringIO()
with contextlib.redirect_stdout(salida_plan):
    con_broadcast.explain()
for linea in salida_plan.getvalue().splitlines():
    if "Join" in linea or "BroadcastExchange" in linea:
        print(linea.strip()[:120])

sin_broadcast = muestra_de_viajes.join(zonas_tipadas, condicion_destino, "left")
print("Mismo número de filas con y sin broadcast:", con_broadcast.count() == sin_broadcast.count())

**Qué observar.** El plan contiene `BroadcastHashJoin` y `BroadcastExchange`: la tabla de zonas viaja completa a cada ejecutor y los viajes se quedan donde están. El broadcast cambia *cómo* se calcula el `join`, no *qué* se calcula, y por eso las dos versiones devuelven el mismo número de filas. Una advertencia: la tabla que envuelves debe caber en la memoria del driver y de cada ejecutor, y `F.broadcast` se la manda a todos aunque sea enorme.

**Pruébalo tú.** Une `muestra_de_viajes` con las zonas por la zona de recogida (`PULocationID`), usando broadcast, y comprueba con un `count` que el `left join` no cambió el número de viajes.

In [ ]:
# Celda 44
# Tu turno: escribe tu código aquí y ejecútala.

<details>
<summary>Una solución posible</summary>

```python
condicion_recogida = muestra_de_viajes["PULocationID"] == zonas_tipadas["LocationID"]
unido = muestra_de_viajes.join(F.broadcast(zonas_tipadas), condicion_recogida, "left")
print(unido.count() == muestra_de_viajes.count())
```

</details>

### 12.6 `dropDuplicates` y `orderBy`: quitar repetidos y ordenar

En el ejercicio 2 del lab de silver y gold de la clase 6 quitas los duplicados con `dropDuplicates(["_hash_fila"])`, que deja una sola fila por cada valor de la lista de columnas. `orderBy` aparece al ordenar un resultado, y se combina con `F.desc` o `F.asc` para elegir el sentido de cada columna. Los dos ya aparecieron en la sección 6; este ejemplo los junta en el patrón de ese lab, con una tabla pequeña cuyas filas repetidas se distinguen solo por el lote en que llegaron.

In [ ]:
# Celda 45
# Tres filas de dos lotes: las dos primeras son el mismo dato (misma huella) ingestado dos veces.
con_repetidos = spark.createDataFrame(
    [("2024-01-01", "h1", "lote_1"), ("2024-01-01", "h1", "lote_2"), ("2024-01-02", "h2", "lote_1")],
    "fecha string, _hash_fila string, _id_lote string",
)

sin_repetidos = con_repetidos.dropDuplicates(["_hash_fila"])          # una fila por cada huella
print("Filas antes:", con_repetidos.count(), "| después:", sin_repetidos.count())

# Cuál de las dos filas con la misma huella se conserva no está garantizado; aquí da igual porque es el mismo dato.
# El resultado se ordena al final, de la fecha más reciente a la más antigua.
sin_repetidos.orderBy(F.desc("fecha")).show()

**Qué observar.** De tres filas quedan dos: la huella `h1` aparece una sola vez, con el lote que Spark haya encontrado primero. Después `orderBy(F.desc("fecha"))` pone primero la fecha más reciente. Fíjate en el orden en que se hacen las cosas: primero se reduce y al final se ordena, porque ordenar un resultado pequeño es barato.

**Pruébalo tú.** Con `con_repetidos`, ordena las tres filas por `_id_lote` de mayor a menor y, dentro de cada lote, por `fecha` de menor a mayor. Muéstralas.

In [ ]:
# Celda 46
# Tu turno: escribe tu código aquí y ejecútala.

<details>
<summary>Una solución posible</summary>

```python
con_repetidos.orderBy(F.desc("_id_lote"), F.asc("fecha")).show()
```

</details>

## 13. Cómo depurar paso a paso

**El problema.** En un pipeline de datos casi nunca falla algo con un mensaje claro. Con mucha frecuencia el código corre, no lanza ningún error y el resultado está mal: faltan filas, sobran filas o un promedio sale raro. Y cuando sí hay un error, el mensaje de Spark es largo y asusta. **La idea de ingeniería** es trabajar de una forma que haga visibles los problemas antes de que se acumulen. Son cuatro hábitos, y los vas a ver en uso en esta sección: escribir el cálculo en pasos con nombre, mirar el resultado de cada paso, probar sobre una muestra pequeña antes de usar todos los datos, y saber leer un error de Spark. **Lo que resuelve** es encontrar *dónde* se rompió algo en lugar de adivinar. **Lo que no resuelve**: ningún hábito reemplaza una prueba escrita; en el lab cada función tiene la suya.

### 13.1 Pasos con nombre y mirar después de cada paso

Una cadena larga de métodos, con un punto detrás de otro, es compacta y difícil de depurar: cuando el resultado está mal, no sabes cuál de los cuatro pasos lo dañó. La pregunta de ejemplo es: ¿cuántos viajes de Uber y de Lyft con propina empiezan en cada distrito? Se resuelve en cuatro pasos, cada uno guardado en una variable con un nombre que dice qué contiene, y la función `mirar` imprime las columnas y unas filas de cada paso. Esas funciones se definen en la Celda 47.

In [ ]:
# Celda 47
def mirar(nombre, df, filas=3):
    """Después de cada paso: el nombre, cuántas columnas tiene y unas filas de muestra."""
    print(f"--- {nombre}: {len(df.columns)} columnas ---")
    df.show(filas, truncate=False)


def mostrar_error(error):
    """Imprime un error de Spark en corto: su clase y las tres primeras líneas del mensaje."""
    print("Clase del error:", type(error).__name__)
    lineas = str(error).strip().splitlines()
    for linea in lineas[:3]:
        print("  ", linea[:300])
    if len(lineas) > 3:
        print(f"   ... ({len(lineas) - 3} líneas más)")


def viajes_con_propina_por_distrito(entrada, ver_pasos=False):
    """Uber (HV0003) y Lyft (HV0005) con propina, contados por distrito de recogida."""
    paso_1_empresas = entrada.where(F.col("hvfhs_license_num").isin("HV0003", "HV0005"))
    if ver_pasos:
        mirar("paso 1: Uber y Lyft", paso_1_empresas.select("hvfhs_license_num", "tips", "PULocationID"))

    paso_2_con_propina = paso_1_empresas.where(F.col("tips") > 0)
    if ver_pasos:
        mirar("paso 2: con propina", paso_2_con_propina.select("hvfhs_license_num", "tips", "PULocationID"))

    condicion_zona = paso_2_con_propina["PULocationID"] == zonas_tipadas["LocationID"]
    paso_3_con_zona = paso_2_con_propina.join(zonas_tipadas, condicion_zona, "left")
    if ver_pasos:
        mirar("paso 3: con su zona", paso_3_con_zona.select("tips", "PULocationID", "Borough"))

    paso_4_por_distrito = (
        paso_3_con_zona
        .groupBy("Borough")
        .agg(F.count(F.lit(1)).alias("viajes"))
        .orderBy(F.desc("viajes"))
    )
    return paso_4_por_distrito

**Qué observar.** Ninguna de estas funciones se ha ejecutado todavía. `mirar` imprime lo que tiene cada paso; `mostrar_error` resume un error de Spark para que quepa en pantalla (las líneas que omite casi siempre son el plan interno de Spark o el seguimiento de Java); y `viajes_con_propina_por_distrito` tiene los cuatro pasos con nombre. El parámetro `ver_pasos` activa las miradas intermedias con un `if`, y así el mismo código sirve para depurar y para producción.

### 13.2 Probar sobre una muestra pequeña

Depurar sobre 19 663 930 filas es lento y cada intento cuesta segundos. Sobre una muestra del 0.1 % cada intento tarda una fracción. `sample(fraction=0.001, seed=1)` toma cada fila con esa probabilidad y la semilla (`seed`) hace que sea siempre la misma muestra, lo que permite repetir un experimento. Es mejor que `limit(n)` para este fin: `limit` toma las primeras filas del archivo, que pertenecen a los primeros minutos de enero y no representan al mes. Cuando el código funciona sobre la muestra, se ejecuta sobre todo. La Celda 48 prueba sobre la muestra y la Celda 49 repite el cálculo sobre todo enero.

In [ ]:
# Celda 48
muestra_depurar = viajes.sample(fraction=0.001, seed=1)
print("Viajes en la muestra:", fmt(muestra_depurar.count()))

resultado_muestra = viajes_con_propina_por_distrito(muestra_depurar, ver_pasos=True)
print("\nResultado sobre la muestra:")
resultado_muestra.show()

**Qué observar.** Cada paso muestra tres filas: en el paso 1 solo hay códigos `HV0003` y `HV0005`; en el paso 2 todas las propinas son mayores que cero; en el paso 3 aparece el distrito de cada viaje. Si un paso mostrara algo inesperado (propinas en cero, un distrito vacío en todas las filas), ahí estaría el problema. Con el código verificado, se ejecuta sobre todo enero cambiando solo el DataFrame de entrada:

In [ ]:
# Celda 49
with cronometro("el mismo código sobre todo enero"):
    resultado_enero = viajes_con_propina_por_distrito(viajes)
    resultado_enero.show()

### 13.3 Leer un error de Spark

Cuando algo falla, Spark imprime un mensaje largo. No hace falta leerlo todo: casi siempre la información útil está en las primeras líneas. La Celda 50 provoca a propósito un error muy común, pedir una columna que no existe, y la función `mostrar_error` lo resume.

In [ ]:
# Celda 50
try:
    viajes.select("PULocation").show(3)
except Exception as error:
    mostrar_error(error)

Así se lee:

1. **La clase del error** (`AnalysisException`) dice en qué fase falló. Un `AnalysisException` quiere decir que Spark revisó tu plan, antes de leer ningún dato, y encontró algo que no existe o no cuadra: una columna inexistente, un tipo que no sirve para esa operación, una tabla que no está. Es el error más frecuente y el más fácil de arreglar.
2. **El código entre corchetes** (`[UNRESOLVED_COLUMN.WITH_SUGGESTION]`) es el nombre del error y es lo que se busca en la documentación.
3. **La frase siguiente** dice qué pasó, en este caso «una columna con nombre `PULocation` no se puede resolver», y suele traer una sugerencia: «¿quisiste decir `PULocationID`, `DOLocationID`...?».
4. **El resto** es el plan interno de Spark (`Project`, `Relation`...). Al principio puedes ignorarlo.

Otras clases de error cuentan otra historia. Un `Py4JJavaError` significa que algo falló dentro de Spark mientras ejecutaba el trabajo (un archivo que no existe, una conversión imposible): ahí lo útil está en la línea que empieza con `Caused by:`. Un `Py4JError` o un `ValueError` aparece cuando la expresión de Python se escribió mal antes de llegar a Spark. Un error de ortografía se arregla en un minuto; lo difícil es creer que el problema está donde no está, y por eso se mira paso a paso.

### 13.4 Tres errores para arreglar

Las Celdas 52, 54 y 56 tienen un error cada una, y las Celdas 53, 55 y 57 comprueban si lo arreglaste con la función `comprobar` que define la Celda 51. Están envueltas en `try` para que el notebook no se detenga y para que veas el mensaje resumido. En cada una: lee el error, encuentra la causa, corrige la línea y ejecuta la celda otra vez hasta que `comprobar` diga `OK`. Si te atascas, cada error trae una pista y, al final de la sección, la solución.

In [ ]:
# Celda 51
def comprobar(nombre, prueba):
    """Corre la prueba y dice OK, PENDIENTE (todavía no hay resultado) o REVISAR (hay resultado, pero está mal)."""
    try:
        prueba()
    except NameError:
        print(f"{nombre}: PENDIENTE  la variable todavía no existe: la celda anterior no terminó sin error.")
    except AssertionError as pista:
        print(f"{nombre}: REVISAR  {pista}")
    else:
        print(f"{nombre}: OK")

#### Error 1: una columna mal escrita

Quieres los viajes de más de diez millas con su distancia, hora y zona.

**Pista.** Lee la frase «no se puede resolver» y las sugerencias: Spark propone el nombre correcto. Compara letra por letra.

In [ ]:
# Celda 52
try:
    largos = viajes.select("pickup_datetime", "PULocationID", "trip_milles").where(F.col("trip_milles") > 10)
    largos.show(3)
except Exception as error:
    mostrar_error(error)

In [ ]:
# Celda 53
def prueba_error_1():
    columnas = largos.columns
    assert columnas == ["pickup_datetime", "PULocationID", "trip_miles"], f"Las columnas deben ser pickup_datetime, PULocationID y trip_miles y tienes {columnas}."
    assert largos.where(F.col("trip_miles") <= 10).count() == 0, "Hay viajes de 10 millas o menos: revisa el filtro."

comprobar("Error 1", prueba_error_1)

#### Error 2: un tipo que no sirve para la operación

Quieres saber a qué hora del día se piden más viajes, y empiezas por calcular la «hora promedio» de recogida. El resultado debería ser un DataFrame `viajes_por_hora` con una fila por cada hora del día.

**Pista.** El mensaje dice qué tipo espera la función y qué tipo recibió. Un promedio de fechas no tiene sentido; lo que sí sirve es sacar la hora de cada fecha y contar los viajes de cada hora (`F.hour`).

In [ ]:
# Celda 54
try:
    viajes_por_hora = viajes.agg(F.avg("pickup_datetime").alias("hora_promedio"))
    viajes_por_hora.show()
except Exception as error:
    mostrar_error(error)

In [ ]:
# Celda 55
def prueba_error_2():
    columnas = viajes_por_hora.columns
    assert len(columnas) == 2, f"Debe tener dos columnas (la hora y el número de viajes) y tiene {len(columnas)}: {columnas}."
    filas = viajes_por_hora.count()
    assert filas == 24, f"Debe tener 24 filas (una por hora) y tiene {filas}."

comprobar("Error 2", prueba_error_2)

#### Error 3: `&` sin paréntesis

Quieres contar los viajes de más de diez millas **y** con propina.

**Pista.** Python evalúa `&` antes que `>`, así que `10 & F.col("tips")` se calcula primero y no es lo que pensabas. El mensaje de Spark habla de un método `and` que no existe para un número entero: es la forma en que se ve ese error. Pon cada condición entre paréntesis.

In [ ]:
# Celda 56
try:
    largos_con_propina_2 = viajes.where(F.col("trip_miles") > 10 & F.col("tips") > 0)
    cuantos = largos_con_propina_2.count()
    print("Viajes largos con propina:", fmt(cuantos))
except Exception as error:
    mostrar_error(error)

In [ ]:
# Celda 57
def prueba_error_3():
    esperado = viajes.where((F.col("trip_miles") > 10) & (F.col("tips") > 0)).count()
    assert cuantos == esperado, f"El conteo debería ser {esperado} y es {cuantos}."

comprobar("Error 3", prueba_error_3)

Cuando los tres digan `OK`, puedes abrir las soluciones para comparar con lo que escribiste.

<details>
<summary>Soluciones de los tres errores</summary>

**Error 1.** El nombre correcto es `trip_miles` (con una sola `l`). Spark ya lo sugería en el mensaje:

```python
largos = viajes.select("pickup_datetime", "PULocationID", "trip_miles").where(F.col("trip_miles") > 10)
```

**Error 2.** Un promedio exige números; una fecha no lo es. Para saber la hora más pedida se extrae la hora y se cuentan los viajes de cada una:

```python
viajes_por_hora = (
    viajes
    .groupBy(F.hour("pickup_datetime").alias("hora"))
    .agg(F.count(F.lit(1)).alias("viajes"))
    .orderBy("hora")
)
```

**Error 3.** Cada condición va entre paréntesis para que `&` una dos condiciones y no un número con una columna:

```python
largos_con_propina_2 = viajes.where((F.col("trip_miles") > 10) & (F.col("tips") > 0))
```

</details>

## 14. Lo que ya sabes hacer

Con las operaciones de este notebook se puede escribir casi todo silver y gold. Esta tabla las resume con la sección donde las usaste:

| Para... | Se usa | Sección |
|---|---|---|
| Ver filas, columnas y tipos | `show`, `limit`, `columns`, `printSchema`, `count` | 2 |
| Elegir columnas | `select`, `F.col`, `drop` | 3 |
| Crear y convertir columnas | `withColumn`, `cast`, `alias`, `withColumnRenamed`, `F.when` | 4 |
| Descartar filas | `where` con `&`, `\|`, `~`, `isNull`, `isNotNull` | 5 |
| Ordenar y quitar repetidos | `orderBy`, `distinct`, `dropDuplicates` | 6 |
| Resumir | `groupBy().agg()` con `F.count`, `F.sum`, `F.avg`, `F.max` | 7 |
| Combinar tablas | `join` (`left`, `inner`, `left_anti`) y la cardinalidad | 8 |
| Escribir lo mismo en SQL | `createOrReplaceTempView`, `spark.sql` | 9 |
| Saber cuándo se ejecuta algo | transformaciones y acciones | 10 |
| Traer resultados a Python | `take`, `collect`, `toPandas` (solo resultados pequeños) | 11 |
| Escribir los ejercicios del lab | `F.coalesce`, `F.lit`, `F.concat_ws`, `F.sha2`, `F.concat`, `F.broadcast`, `dropDuplicates`, `orderBy` | 12 |
| Encontrar un error | pasos con nombre, `mirar`, muestra, leer el mensaje | 13 |

Falta explicar qué hace Spark por dentro cuando ejecuta estas líneas: cómo reparte los datos, por qué `groupBy` y `join` son más costosos que `where`, y qué es el *shuffle*. Eso es el notebook `02_como_funciona.ipynb`. El lab, `03_lab.ipynb`, usa todo lo de este notebook para construir tu pipeline.

Antes de abrir otro notebook, apaga Spark con la Celda 58 (solo cabe un Spark a la vez en esta máquina).

In [ ]:
# Celda 58
spark.stop()
print("Spark cerrado. Ya puedes abrir el siguiente notebook.")